# Advanced Model Comparison and Selection\n\n**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**\n\n## Learning Objectives\n\n- Master systematic model comparison frameworks\n- Use information criteria (AIC, BIC) effectively\n- Implement nested and non-nested model tests\n- Perform robust cross-validation strategies\n- Build model ensembles for better predictions\n- Understand the bias-variance tradeoff\n- Select models based on business objectives

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet\nfrom sklearn.neighbors import KNeighborsRegressor\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor\nfrom sklearn.model_selection import train_test_split, cross_val_score, KFold, TimeSeriesSplit\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error\nimport statsmodels.api as sm\nimport statsmodels.formula.api as smf\nfrom scipy import stats\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (14, 8)

## The Model Selection Problem\n\n**Question:** How do we choose the best model?\n\n**Considerations:**\n1. **Predictive accuracy** - How well does it generalize?\n2. **Interpretability** - Can we explain the predictions?\n3. **Computational cost** - How fast is training/prediction?\n4. **Robustness** - How sensitive to outliers/noise?\n5. **Business constraints** - What matters to stakeholders?\n\n**No free lunch:** No single model is best for all problems!

## Load Data

In [ ]:
# Generate comprehensive match dataset\nnp.random.seed(42)\nn = 500\n\ndf = pd.DataFrame({\n    'shots': np.random.randint(5, 20, n),\n    'shots_on_target': np.random.randint(2, 12, n),\n    'possession': np.random.uniform(35, 70, n),\n    'pass_accuracy': np.random.uniform(70, 92, n),\n    'xg': np.random.uniform(0.5, 3.5, n),\n    'opponent_xg': np.random.uniform(0.5, 3.5, n),\n    'home': np.random.choice([0, 1], n),\n    'opponent_strength': np.random.uniform(0.3, 0.9, n)\n})\n\n# Generate realistic goals\ndf['goals'] = (\n    df['xg'] * 0.75 + \n    df['shots_on_target'] * 0.08 + \n    df['home'] * 0.3 - \n    df['opponent_strength'] * 0.4 +\n    (df['possession'] - 50) * 0.01 +\n    np.random.normal(0, 0.5, n)\n).clip(0, 6).round().astype(int)\n\nprint(f\"Dataset: {df.shape}\")\nprint(df.head())\n\n# Prepare features and target\nfeature_cols = ['shots', 'shots_on_target', 'possession', 'pass_accuracy', \n                'xg', 'opponent_xg', 'home', 'opponent_strength']\nX = df[feature_cols]\ny = df['goals']\n\n# Train/test split\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)\n\nprint(f\"\\nTrain: {len(X_train)}, Test: {len(X_test)}\")

## 1. Information Criteria: AIC and BIC\n\n**AIC (Akaike Information Criterion):**\n- Balances fit and complexity\n- Penalizes number of parameters\n- Lower is better\n\n**BIC (Bayesian Information Criterion):**\n- Similar to AIC but stronger penalty for complexity\n- Prefers simpler models\n- Lower is better\n\n**When to use:**\n- Comparing models with different numbers of features\n- Non-nested models\n- Want to avoid overfitting

In [ ]:
# Fit models with different feature sets using statsmodels\nmodels_ic = {}\n\n# Model 1: Basic features\nformula1 = 'goals ~ xg + shots_on_target + home'\nmodels_ic['Basic'] = smf.ols(formula1, data=pd.concat([X_train, y_train], axis=1)).fit()\n\n# Model 2: Extended features\nformula2 = 'goals ~ xg + shots_on_target + home + opponent_strength + possession'\nmodels_ic['Extended'] = smf.ols(formula2, data=pd.concat([X_train, y_train], axis=1)).fit()\n\n# Model 3: All features\nformula3 = 'goals ~ ' + ' + '.join(feature_cols)\nmodels_ic['Full'] = smf.ols(formula3, data=pd.concat([X_train, y_train], axis=1)).fit()\n\n# Compare using AIC and BIC\ncomparison = []\nfor name, model in models_ic.items():\n    comparison.append({\n        'Model': name,\n        'Features': model.df_model,\n        'R²': model.rsquared,\n        'Adj R²': model.rsquared_adj,\n        'AIC': model.aic,\n        'BIC': model.bic\n    })\n\ncomparison_df = pd.DataFrame(comparison)\nprint(\"Information Criteria Comparison:\")\nprint(comparison_df.to_string(index=False))\n\nbest_aic = comparison_df.loc[comparison_df['AIC'].idxmin(), 'Model']\nbest_bic = comparison_df.loc[comparison_df['BIC'].idxmin(), 'Model']\n\nprint(f\"\\nBest by AIC: {best_aic}\")\nprint(f\"Best by BIC: {best_bic}\")\nprint(f\"\\nNote: BIC typically prefers simpler models than AIC.\")

## 2. Cross-Validation Strategies\n\n**Standard K-Fold:** Random splits\n**Stratified K-Fold:** Preserves class distribution\n**Time Series Split:** Respects temporal order\n**Leave-One-Out:** Each observation is test set once

In [ ]:
# Compare different CV strategies\nmodel = LinearRegression()\n\n# 1. Standard K-Fold\nkfold = KFold(n_splits=5, shuffle=True, random_state=42)\nscores_kfold = cross_val_score(model, X, y, cv=kfold, scoring='r2')\n\n# 2. Time Series Split (assuming data is ordered by time)\ntscv = TimeSeriesSplit(n_splits=5)\nscores_tscv = cross_val_score(model, X, y, cv=tscv, scoring='r2')\n\nprint(\"Cross-Validation Strategy Comparison:\")\nprint(f\"\\nStandard K-Fold (5-fold):\")\nprint(f\"  Mean R²: {scores_kfold.mean():.3f} (±{scores_kfold.std():.3f})\")\nprint(f\"  Scores: {scores_kfold}\")\n\nprint(f\"\\nTime Series Split (5-fold):\")\nprint(f\"  Mean R²: {scores_tscv.mean():.3f} (±{scores_tscv.std():.3f})\")\nprint(f\"  Scores: {scores_tscv}\")\n\nprint(f\"\\nNote: Time Series Split is more realistic for temporal data.\")\nprint(f\"It trains on past data and tests on future data.\")

## 3. Comprehensive Model Comparison\n\nLet's compare multiple model types systematically.

In [ ]:
# Scale features for models that need it\nscaler = StandardScaler()\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)\n\n# Define models\nmodels = {\n    'Linear Regression': LinearRegression(),\n    'Ridge (α=1.0)': Ridge(alpha=1.0),\n    'Lasso (α=0.1)': Lasso(alpha=0.1),\n    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),\n    'KNN (K=7)': KNeighborsRegressor(n_neighbors=7),\n    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),\n    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),\n    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)\n}\n\n# Evaluate each model\nresults = []\n\nfor name, model in models.items():\n    # Use scaled data for distance-based and regularized models\n    use_scaled = name in ['Ridge (α=1.0)', 'Lasso (α=0.1)', 'ElasticNet', 'KNN (K=7)']\n    X_tr = X_train_scaled if use_scaled else X_train\n    X_te = X_test_scaled if use_scaled else X_test\n    \n    # Train\n    model.fit(X_tr, y_train)\n    \n    # Predict\n    y_train_pred = model.predict(X_tr)\n    y_test_pred = model.predict(X_te)\n    \n    # Metrics\n    train_r2 = r2_score(y_train, y_train_pred)\n    test_r2 = r2_score(y_test, y_test_pred)\n    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))\n    test_mae = mean_absolute_error(y_test, y_test_pred)\n    \n    # Cross-validation\n    X_cv = scaler.fit_transform(X) if use_scaled else X\n    cv_scores = cross_val_score(model, X_cv, y, cv=5, scoring='r2')\n    \n    results.append({\n        'Model': name,\n        'Train R²': train_r2,\n        'Test R²': test_r2,\n        'CV R² (mean)': cv_scores.mean(),\n        'CV R² (std)': cv_scores.std(),\n        'Test RMSE': test_rmse,\n        'Test MAE': test_mae,\n        'Overfit Gap': train_r2 - test_r2\n    })\n\nresults_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)\n\nprint(\"Comprehensive Model Comparison:\")\nprint(results_df.to_string(index=False))\n\nprint(f\"\\nBest model by Test R²: {results_df.iloc[0]['Model']}\")\nprint(f\"Best model by CV R²: {results_df.loc[results_df['CV R² (mean)'].idxmax(), 'Model']}\")\nprint(f\"Least overfitting: {results_df.loc[results_df['Overfit Gap'].idxmin(), 'Model']}\")

## Visualize Model Comparison

In [ ]:
# Create comprehensive visualization\nfig, axes = plt.subplots(2, 2, figsize=(16, 12))\n\n# 1. Train vs Test R²\naxes[0, 0].scatter(results_df['Train R²'], results_df['Test R²'], s=100, alpha=0.7)\nfor idx, row in results_df.iterrows():\n    axes[0, 0].annotate(row['Model'], (row['Train R²'], row['Test R²']), \n                       fontsize=8, ha='right')\naxes[0, 0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Generalization')\naxes[0, 0].set_xlabel('Train R²')\naxes[0, 0].set_ylabel('Test R²')\naxes[0, 0].set_title('Train vs Test Performance')\naxes[0, 0].legend()\naxes[0, 0].grid(True, alpha=0.3)\n\n# 2. Test R² with error bars (CV std)\naxes[0, 1].barh(results_df['Model'], results_df['Test R²'], alpha=0.7)\naxes[0, 1].errorbar(results_df['Test R²'], range(len(results_df)), \n                    xerr=results_df['CV R² (std)'], fmt='none', color='red', capsize=5)\naxes[0, 1].set_xlabel('Test R²')\naxes[0, 1].set_title('Test R² with CV Uncertainty')\naxes[0, 1].grid(True, alpha=0.3, axis='x')\n\n# 3. RMSE comparison\naxes[1, 0].bar(range(len(results_df)), results_df['Test RMSE'], alpha=0.7, color='coral')\naxes[1, 0].set_xticks(range(len(results_df)))\naxes[1, 0].set_xticklabels(results_df['Model'], rotation=45, ha='right')\naxes[1, 0].set_ylabel('Test RMSE')\naxes[1, 0].set_title('Test RMSE (Lower is Better)')\naxes[1, 0].grid(True, alpha=0.3, axis='y')\n\n# 4. Overfitting gap\ncolors = ['green' if x < 0.05 else 'orange' if x < 0.1 else 'red' for x in results_df['Overfit Gap']]\naxes[1, 1].barh(results_df['Model'], results_df['Overfit Gap'], color=colors, alpha=0.7)\naxes[1, 1].axvline(x=0.05, color='orange', linestyle='--', label='Acceptable (<0.05)')\naxes[1, 1].axvline(x=0.1, color='red', linestyle='--', label='Concerning (>0.1)')\naxes[1, 1].set_xlabel('Overfit Gap (Train R² - Test R²)')\naxes[1, 1].set_title('Overfitting Analysis')\naxes[1, 1].legend()\naxes[1, 1].grid(True, alpha=0.3, axis='x')\n\nplt.tight_layout()\nplt.show()\n\nprint(\"Interpretation:\")\nprint(\"- Top-left: Points near diagonal = good generalization\")\nprint(\"- Top-right: Longer error bars = higher variance\")\nprint(\"- Bottom-left: Lower bars = better predictions\")\nprint(\"- Bottom-right: Green = minimal overfitting, Red = severe overfitting\")

## 4. Model Ensembles\n\n**Idea:** Combine multiple models for better predictions.\n\n**Methods:**\n1. **Simple averaging:** Average predictions from all models\n2. **Weighted averaging:** Weight by performance\n3. **Stacking:** Train a meta-model on predictions

In [ ]:
# Create ensemble predictions\nensemble_models = {\n    'Linear': LinearRegression(),\n    'Ridge': Ridge(alpha=1.0),\n    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)\n}\n\n# Train models and get predictions\npredictions = {}\nfor name, model in ensemble_models.items():\n    use_scaled = name == 'Ridge'\n    X_tr = X_train_scaled if use_scaled else X_train\n    X_te = X_test_scaled if use_scaled else X_test\n    \n    model.fit(X_tr, y_train)\n    predictions[name] = model.predict(X_te)\n\n# 1. Simple averaging\nensemble_simple = np.mean(list(predictions.values()), axis=0)\n\n# 2. Weighted averaging (by test R²)\nweights = []\nfor name in ensemble_models.keys():\n    pred = predictions[name]\n    r2 = r2_score(y_test, pred)\n    weights.append(r2)\n\nweights = np.array(weights) / np.sum(weights)  # Normalize\nensemble_weighted = np.average(list(predictions.values()), axis=0, weights=weights)\n\n# Evaluate ensembles\nprint(\"Ensemble Performance:\")\nprint(f\"\\nIndividual Models:\")\nfor name, pred in predictions.items():\n    r2 = r2_score(y_test, pred)\n    rmse = np.sqrt(mean_squared_error(y_test, pred))\n    print(f\"  {name}: R² = {r2:.3f}, RMSE = {rmse:.3f}\")\n\nprint(f\"\\nEnsembles:\")\nr2_simple = r2_score(y_test, ensemble_simple)\nrmse_simple = np.sqrt(mean_squared_error(y_test, ensemble_simple))\nprint(f\"  Simple Average: R² = {r2_simple:.3f}, RMSE = {rmse_simple:.3f}\")\n\nr2_weighted = r2_score(y_test, ensemble_weighted)\nrmse_weighted = np.sqrt(mean_squared_error(y_test, ensemble_weighted))\nprint(f\"  Weighted Average: R² = {r2_weighted:.3f}, RMSE = {rmse_weighted:.3f}\")\n\nif r2_weighted > max([r2_score(y_test, p) for p in predictions.values()]):\n    print(f\"\\n✅ Ensemble outperforms individual models!\")\nelse:\n    print(f\"\\n⚠️ Best individual model still performs better.\")

## 5. Bias-Variance Tradeoff\n\n**Bias:** Error from wrong assumptions (underfitting)\n**Variance:** Error from sensitivity to training data (overfitting)\n\n**Goal:** Find the sweet spot!

In [ ]:
# Visualize bias-variance tradeoff with polynomial features\nfrom sklearn.preprocessing import PolynomialFeatures\nfrom sklearn.pipeline import make_pipeline\n\n# Use single feature for visualization\nX_simple = df[['xg']].values\ny_simple = df['goals'].values\n\nX_train_s, X_test_s, y_train_s, y_test_s = train_test_split(\n    X_simple, y_simple, test_size=0.25, random_state=42\n)\n\n# Test different polynomial degrees\ndegrees = range(1, 11)\ntrain_scores = []\ntest_scores = []\n\nfor degree in degrees:\n    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())\n    model.fit(X_train_s, y_train_s)\n    \n    train_scores.append(model.score(X_train_s, y_train_s))\n    test_scores.append(model.score(X_test_s, y_test_s))\n\n# Plot\nplt.figure(figsize=(12, 6))\nplt.plot(degrees, train_scores, 'o-', label='Training R²', linewidth=2)\nplt.plot(degrees, test_scores, 's-', label='Test R²', linewidth=2)\nplt.xlabel('Polynomial Degree (Model Complexity)')\nplt.ylabel('R² Score')\nplt.title('Bias-Variance Tradeoff')\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.axvline(x=degrees[np.argmax(test_scores)], color='green', linestyle='--', \n            label=f'Optimal Complexity (degree={degrees[np.argmax(test_scores)]})')\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint(\"Interpretation:\")\nprint(\"- Low degree: High bias (underfitting) - both scores low\")\nprint(\"- Optimal degree: Best test performance\")\nprint(\"- High degree: High variance (overfitting) - train high, test low\")\nprint(f\"\\nOptimal polynomial degree: {degrees[np.argmax(test_scores)]}\")

## 6. Model Selection Framework\n\nA systematic approach to choosing models.

In [ ]:
def model_selection_framework(results_df, priorities):\n    \"\"\"\n    Select best model based on priorities.\n    \n    priorities: dict with keys 'accuracy', 'interpretability', 'speed', 'robustness'\n    Each value is a weight from 0 to 1 (should sum to 1)\n    \"\"\"\n    \n    # Define model characteristics (0-1 scale)\n    characteristics = {\n        'Linear Regression': {'accuracy': 0.7, 'interpretability': 1.0, 'speed': 1.0, 'robustness': 0.6},\n        'Ridge (α=1.0)': {'accuracy': 0.75, 'interpretability': 0.9, 'speed': 0.9, 'robustness': 0.7},\n        'Lasso (α=0.1)': {'accuracy': 0.75, 'interpretability': 0.95, 'speed': 0.9, 'robustness': 0.7},\n        'ElasticNet': {'accuracy': 0.75, 'interpretability': 0.9, 'speed': 0.9, 'robustness': 0.75},\n        'KNN (K=7)': {'accuracy': 0.8, 'interpretability': 0.4, 'speed': 0.5, 'robustness': 0.5},\n        'Decision Tree': {'accuracy': 0.75, 'interpretability': 0.8, 'speed': 0.8, 'robustness': 0.4},\n        'Random Forest': {'accuracy': 0.85, 'interpretability': 0.5, 'speed': 0.6, 'robustness': 0.8},\n        'Gradient Boosting': {'accuracy': 0.9, 'interpretability': 0.4, 'speed': 0.5, 'robustness': 0.7}\n    }\n    \n    # Normalize accuracy by test R²\n    max_r2 = results_df['Test R²'].max()\n    for model_name in characteristics.keys():\n        if model_name in results_df['Model'].values:\n            actual_r2 = results_df[results_df['Model'] == model_name]['Test R²'].values[0]\n            characteristics[model_name]['accuracy'] = actual_r2 / max_r2\n    \n    # Calculate weighted scores\n    scores = {}\n    for model_name, chars in characteristics.items():\n        score = sum(chars[key] * priorities[key] for key in priorities.keys())\n        scores[model_name] = score\n    \n    # Rank models\n    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)\n    \n    return ranked\n\n# Example 1: Prioritize accuracy\nprint(\"Scenario 1: Research Project (Accuracy Priority)\")\npriorities_accuracy = {\n    'accuracy': 0.7,\n    'interpretability': 0.1,\n    'speed': 0.1,\n    'robustness': 0.1\n}\nranked_accuracy = model_selection_framework(results_df, priorities_accuracy)\nprint(\"Top 3 models:\")\nfor i, (model, score) in enumerate(ranked_accuracy[:3], 1):\n    print(f\"  {i}. {model}: {score:.3f}\")\n\n# Example 2: Prioritize interpretability\nprint(\"\\nScenario 2: Stakeholder Presentation (Interpretability Priority)\")\npriorities_interpret = {\n    'accuracy': 0.3,\n    'interpretability': 0.5,\n    'speed': 0.1,\n    'robustness': 0.1\n}\nranked_interpret = model_selection_framework(results_df, priorities_interpret)\nprint(\"Top 3 models:\")\nfor i, (model, score) in enumerate(ranked_interpret[:3], 1):\n    print(f\"  {i}. {model}: {score:.3f}\")\n\n# Example 3: Balanced\nprint(\"\\nScenario 3: Production System (Balanced)\")\npriorities_balanced = {\n    'accuracy': 0.4,\n    'interpretability': 0.2,\n    'speed': 0.2,\n    'robustness': 0.2\n}\nranked_balanced = model_selection_framework(results_df, priorities_balanced)\nprint(\"Top 3 models:\")\nfor i, (model, score) in enumerate(ranked_balanced[:3], 1):\n    print(f\"  {i}. {model}: {score:.3f}\")

## 7. Statistical Model Comparison Tests\n\n**For nested models:** Use F-test or likelihood ratio test\n**For non-nested models:** Use AIC/BIC or cross-validation

In [ ]:
# Nested model comparison using F-test\ntrain_data = pd.concat([X_train, y_train], axis=1)\n\n# Reduced model\nmodel_reduced = smf.ols('goals ~ xg + home', data=train_data).fit()\n\n# Full model\nmodel_full = smf.ols('goals ~ ' + ' + '.join(feature_cols), data=train_data).fit()\n\n# F-test\nf_test = model_reduced.compare_f_test(model_full)\n\nprint(\"Nested Model Comparison (F-test):\")\nprint(f\"\\nReduced Model: goals ~ xg + home\")\nprint(f\"  R²: {model_reduced.rsquared:.3f}\")\nprint(f\"  Parameters: {model_reduced.df_model}\")\n\nprint(f\"\\nFull Model: goals ~ all features\")\nprint(f\"  R²: {model_full.rsquared:.3f}\")\nprint(f\"  Parameters: {model_full.df_model}\")\n\nprint(f\"\\nF-statistic: {f_test[0][0]:.3f}\")\nprint(f\"p-value: {f_test[1]:.4f}\")\n\nif f_test[1] < 0.05:\n    print(\"\\n✅ Full model is significantly better than reduced model.\")\nelse:\n    print(\"\\n⚠️ Additional features do not significantly improve the model.\")\n    print(\"Consider using the simpler model (Occam's Razor).\")

## Summary\n\nIn this notebook, we:\n\n1. ✅ Used information criteria (AIC, BIC) for model selection\n2. ✅ Implemented multiple cross-validation strategies\n3. ✅ Compared 8 different regression models systematically\n4. ✅ Built model ensembles for improved predictions\n5. ✅ Visualized the bias-variance tradeoff\n6. ✅ Created a model selection framework based on priorities\n7. ✅ Performed statistical tests for nested models\n\n## Key Takeaways\n\n- **No single best model** - depends on context and priorities\n- **AIC/BIC** help compare models with different complexity\n- **Cross-validation** provides robust performance estimates\n- **Ensembles** often outperform individual models\n- **Bias-variance tradeoff** guides model complexity\n- **Business objectives** should drive model selection\n- **Statistical tests** formalize model comparisons\n- **Interpretability** matters for stakeholder buy-in\n\n## Model Selection Checklist\n\n✅ Define success criteria (accuracy, interpretability, speed)\n✅ Split data properly (train/validation/test)\n✅ Compare multiple model types\n✅ Use cross-validation for robust estimates\n✅ Check for overfitting (train vs test gap)\n✅ Consider ensemble methods\n✅ Validate on holdout test set\n✅ Document model limitations\n✅ Monitor performance in production\n✅ Update models regularly with new data

## Practice Exercises\n\n1. **Nested Cross-Validation**: Implement nested CV for hyperparameter tuning\n2. **Bayesian Model Selection**: Use Bayesian methods for model comparison\n3. **Online Learning**: Implement incremental model updates\n4. **A/B Testing**: Design an A/B test framework for model comparison in production\n5. **Cost-Sensitive Learning**: Incorporate prediction costs into model selection\n6. **Multi-Objective Optimization**: Balance multiple objectives (accuracy, fairness, speed)\n7. **AutoML**: Explore automated model selection tools